In [16]:
from qiskit_braket_provider import AWSBraketProvider, BraketLocalBackend, BraketSampler
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, balanced_accuracy_score
import pandas as pd
import numpy as np

from braket.aws import AwsDevice
# The qiskit_braket_provider creates a seamless connection to avoid collisions.
# Using BraketLocalBackend to run the exact same tasks completely free locally.
backend = BraketLocalBackend()


In [17]:
df = pd.read_csv('../data/cleaned/wildfire_cleaned.csv')

In [18]:
df.head()

,zip,year,fire_occurred,avg_tmax_c,avg_tmin_c,tot_prcp_mm,prev_year_fire,prev_year_acres,rolling_3yr_fire_count
0,85364,2020,1,23.938875,10.715056,241.85,0,0.0,0.0
1,89029,2020,1,23.938875,10.715056,241.85,0,0.0,0.0
2,89029,2022,0,23.938875,10.715056,241.85,0,0.0,1.0
3,89439,2019,1,23.938875,10.715056,241.85,0,0.0,0.0
4,89439,2023,0,23.938875,10.715056,241.85,0,0.0,0.0


In [19]:
test_2023 = df[df["year"] == 2023] # Saving 2023 data to be compared later
df = df[df["year"].between(2018, 2022)].reset_index(drop=True)

assert "fire_occurred" in df.columns, "Missing target column!"
assert df["fire_occurred"].isin([0, 1]).all(), "Target column has non-binary values!"
assert df.duplicated(subset=["year", "zip"]).sum() == 0, "Duplicate Year+ZIP rows found!"
assert df["avg_tmax_c"].isnull().sum() == 0, "Null values found in weather features!"

print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
print("Unique ZIPs:", df["zip"].nunique())
print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:")
print(df["fire_occurred"].value_counts())
print(f"Class imbalance ratio: {df['fire_occurred'].value_counts()[0] / df['fire_occurred'].value_counts()[1]:.1f}:1")

Shape: (3650, 9)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Unique ZIPs: 2009
Nulls:
 Series([], dtype: int64)

Target distribution:
fire_occurred
0    2599
1    1051
Name: count, dtype: int64
Class imbalance ratio: 2.5:1


In [20]:
df["temp_range"] = df["avg_tmax_c"] - df["avg_tmin_c"]  # daily temperature swing

print("Final features:")
print(df.drop(columns=["fire_occurred"]).columns.tolist())
print("\nShape:", df.shape)

Final features:
['zip', 'year', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm', 'prev_year_fire', 'prev_year_acres', 'rolling_3yr_fire_count', 'temp_range']

Shape: (3650, 10)


In [21]:
train, test = train_test_split(df, test_size=0.2, stratify=df["fire_occurred"], random_state=42)

X_train = train.drop(columns=["fire_occurred"])
y_train = train["fire_occurred"]

X_test = test.drop(columns=["fire_occurred"])
y_test = test["fire_occurred"]

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

Train: (2920, 9) {0: 2079, 1: 841}
Test: (730, 9) {0: 520, 1: 210}


In [22]:
FEATURES = [c for c in df.columns if c not in ["zip", "year", "fire_occurred"]]
X_train, y_train = train[FEATURES].values, train["fire_occurred"].values
X_test,  y_test  = test[FEATURES].values,  test["fire_occurred"].values

print("\nTrain shape:", X_train.shape)
print("Test shape: ", X_test.shape)


Train shape: (2920, 7)
Test shape:  (730, 7)


In [23]:
# RandomForest to rank feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importances:\n", importances)

TOP_N = 6  # capped at 6 — new dataset has 7 features total
top_features = importances.head(TOP_N).index.tolist()
print(f"\nTop {TOP_N} features selected:", top_features)
X_train = train[top_features].values
X_test  = test[top_features].values

# Normalize to [-π, π] for quantum encoding
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nFeature range after scaling:")
print("  Min:", X_train.min().round(3), " Max:", X_train.max().round(3))


Feature importances:
 avg_tmin_c                0.231286
avg_tmax_c                0.194890
temp_range                0.185883
tot_prcp_mm               0.151565
rolling_3yr_fire_count    0.139381
prev_year_acres           0.065233
prev_year_fire            0.031761
dtype: float64

Top 6 features selected: ['avg_tmin_c', 'avg_tmax_c', 'temp_range', 'tot_prcp_mm', 'rolling_3yr_fire_count', 'prev_year_acres']

Feature range after scaling:
  Min: -3.142  Max: 3.142


In [24]:
# This is the actual quantum portion of our code. we're using a ZZFeatureMap, which basically encodes our feature columns into
# their quantum states (converting their values into quantum bits).

NUM_QUBITS = 6  # matches TOP_N above

feature_map = ZZFeatureMap(
    feature_dimension=6,
    reps=2,
    entanglement="linear"
)

print(f"Number of qubits : {feature_map.num_qubits}")
print(f"Circuit depth    : {feature_map.decompose().depth()}")
print(f"Number of params : {feature_map.num_parameters}")
print()
feature_map.decompose().draw("text")

# Ok this circuit looks very confusing, but it's just showing the encoding process of our features.

Number of qubits : 6
Circuit depth    : 25
Number of params : 6



C:\Users\cwang\AppData\Local\Temp\ipykernel_29656\3096904475.py:6: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


┌───┐┌───────────┐                                        ┌───┐»
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■──┤ H ├»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐└───┘»
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──■──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«             ┌───────────┐                                                 »
«q_0: ────────┤ P(2*x[0]) ├─────────────────────────────────────────────────»
«             └───────────┘              ┌───┐        ┌───────────┐         »
«q_1: ────────────────────────────────■──┤ H ├────────┤ P(2*x[1]) ├─────────»
«     ┌────────────────────────────┐┌─┴─┐└───┘        └───────────┘         »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──■────────────────────────────────»
«     └────────────────────────────┘└───┘┌─┴─┐┌────────────────────────────┐»
«q_3: ───────────────────────────────────┤ X ├┤ P(2*(π - x[2])*(π - x[3])) ├»
«                                        └───┘└────────────────────────────┘»
«q_4: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_5: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«                                                                           »
«q_0: ──■──────────────────────────────────────────────■────────────────────»
«     ┌─┴─┐┌────────────────────────────┐            ┌─┴─┐                  »
«q_1: ┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├────────────┤ X ├───────────────■──»
«     └───┘└───────────┬───┬────────────┘        ┌───┴───┴───┐         ┌─┴─┐»
«q_2: ──■──────────────┤ H ├─────────────────────┤ P(2*x[2]) ├─────────┤ X ├»
«     ┌─┴─┐            └───┘                     └───────────┘         └───┘»
«q_3: ┤ X ├──────────────■───────────────────────────────────────────────■──»
«     └───┘            ┌─┴─┐             ┌────────────────────────────┐┌─┴─┐»
«q_4: ─────────────────┤ X ├─────────────┤ P(2*(π - x[3])*(π - x[4])) ├┤ X ├»
«                      └───┘             └────────────────────────────┘└───┘»
«q_5: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«                                                                      »
«q_0: ─────────────────────────────────────────────────────────────────»
«                                                                      »
«q_1: ────────────────────────────────────────────■────────────────────»
«     ┌────────────────────────────┐            ┌─┴─┐                  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├────────────┤ X ├───────────────■──»
«     └───────────┬───┬────────────┘        ┌───┴───┴───┐         ┌─┴─┐»
«q_3: ────────────┤ H ├─────────────────────┤ P(2*x[3]) ├─────────┤ X ├»
«                 └───┘                     └───────────┘         └───┘»
«q_4: ──────────────■───────────────────────────────────────────────■──»
«                 ┌─┴─┐             ┌────────────────────────────┐┌─┴─┐»
«q_5: ────────────┤ X ├─────────────┤ P(2*(π - x[4])*(π - x[5])) ├┤ X ├»
«                 └───┘             └────────────────────────────┘└───┘»
«                                                     »
«q_0: ────────────────────────────────────────────────»
«     

In [25]:
# A sampler is just the backend that actually runs the quantum circuit. it's just like how we have to select a kernel
# to run Python.
sampler = BraketSampler(backend=backend)
fidelity = ComputeUncompute(sampler = sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map = feature_map,
    fidelity = fidelity
)

# SVC stands for Support Vector Classifier. It's a classic ML model which essentially looks at our data and finds the best boundary line
# or (divider) that separates the two classes that we're checking for (in our case, wildfire or no wildfire)

model = SVC(
    kernel = 'rbf',
    #kernel = quantum_kernel.evaluate,
    probability = False, # By making this true, we make the model output a probability score instead of just yes/no
    class_weight = 'balanced',
    random_state = 42
)

quantum_model = SVC(
    kernel = quantum_kernel.evaluate,
    probability = True, # By making this true, we make the model output a probability score instead of just yes/no
    class_weight = 'balanced',
    random_state = 42
)

In [26]:
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [27]:
conf_matrix = confusion_matrix(y_test, pred)
class_report = classification_report(y_test, pred, zero_division = 0)
balanced_accuracy = balanced_accuracy_score(y_test, pred)
fl_weighted = f1_score(y_test, pred, average = 'weighted')
f1_macro = f1_score(y_test, pred, average = 'macro')

print(class_report)
print(f'Balanced Accuracy: {balanced_accuracy}')
print(f'F1-Weighted: {fl_weighted}')
print(f'F1-Macro: {f1_macro}')
print(conf_matrix)

              precision    recall  f1-score   support

           0       0.86      0.86      0.86       520
           1       0.65      0.64      0.65       210

    accuracy                           0.80       730
   macro avg       0.75      0.75      0.75       730
weighted avg       0.80      0.80      0.80       730

Balanced Accuracy: 0.7512362637362637
F1-Weighted: 0.7969698185798609
F1-Macro: 0.7519492327048647
[[447  73]
 [ 75 135]]


In [28]:
quantum_model.fit(X_train, y_train)
quantum_pred = quantum_model.predict(X_test)
quantum_prob = quantum_model.predict_proba(X_test)[:, 1]

KeyboardInterrupt: 

In [ ]:
conf_matrix = confusion_matrix(y_test, quantum_pred)
class_report = classification_report(y_test, quantum_pred, zero_division = 0)
balanced_accuracy = balanced_accuracy_score(y_test, quantum_pred)
fl_weighted = f1_score(y_test, quantum_pred, average = 'weighted')
f1_macro = f1_score(y_test, quantum_pred, average = 'macro')
roc_auc = roc_auc_score(y_test, quantum_prob)

print(class_report)
print(f"ROC-AUC: {roc_auc}")
print(f'Balanced Accuracy: {balanced_accuracy}')
print(f'F1-Weighted: {fl_weighted}')
print(f'F1-Macro: {f1_macro}')

print(conf_matrix)

              precision    recall  f1-score   support

           0       0.87      0.84      0.85       631
           1       0.56      0.61      0.58       210

    accuracy                           0.78       841
   macro avg       0.71      0.73      0.72       841
weighted avg       0.79      0.78      0.78       841

Balanced Accuracy: 0.7255263753678968
F1-Weighted: 0.7847169173077835
F1-Macro: 0.7176616552328128
[[528 103]
 [ 81 129]]
